<a href="https://colab.research.google.com/github/prajwalpanditdev/Torrent-To-Google-Drive-Downloader/blob/main/Torrent_To_Google_Drive_Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Torrent To Google Drive Downloader**

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### **Install libtorrent and Initialize Session**

In [0]:
!apt remove -y python3-libtorrent
!pip install --no-cache-dir libtorrent

import libtorrent as lt

params = lt.session_params()
settings = params.settings
settings["listen_interfaces"] = "0.0.0.0:6881,0.0.0.0:6891"

ses = lt.session(params)

downloads = []

### **Mount Google Drive**

In [0]:
from google.colab import drive

drive.mount("/content/drive")

### **Add Torrent From Torrent File**

In [0]:
from google.colab import files

source = files.upload()

params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}

downloads.append(ses.add_torrent(params))

### **Add Torrent From Magnet Link**

In [0]:
while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    
    params = lt.parse_magnet_uri(magnet_link)
    params.save_path = "/content/drive/My Drive/Torrent"
    downloads.append(ses.add_torrent(params))

### **Start Download**

In [0]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}

downloads = [d for d in downloads if d.is_valid()]

download_bars = [
    widgets.FloatSlider(step=0.01, disabled=True, layout=layout, style=style)
    for _ in downloads
]

display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        s = download.status()

        if not s.is_seeding:
            bar.description = " ".join([
                s.name,
                str(s.download_rate / 1000),
                "kB/s",
                state_str[s.state],
            ])
            bar.value = s.progress * 100
        else:
            torrent_name = s.name
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close()
            download_bars.remove(bar)
            print(torrent_name, "complete")
            next_shift -= 1

    time.sleep(1)